<a href="https://colab.research.google.com/github/ameemaiqbal/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ameemaiqbal/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected.")

Connected.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row = one content item, on one day (in fact_content_daily_performance), verified below. A small number of groups (6,390 out of 78,835,655 rows, ~0.008%) violate strict one-row-per-client-content-day grain, likely late-arriving or corrected data. This is negligible for aggregate analysis but means any per-day exact-match query should technically de-duplicate first (e.g. take the latest or sum duplicates) rather than assume uniqueness.
Time window: verified below, the full daily table spans 2025-01-27 to 2026-06-30 (520 distinct days, about 17 months), matching the warehouse's stated coverage. As shown in Notebook 03, this total span isn't evenly available per client, most clients start well after 2025-01-27.

In [9]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Rows violating 1-row-per-client-content-day grain: {len(grain_check)}")

date_range = con.sql(f"""
    SELECT MIN(report_date) AS earliest, MAX(report_date) AS latest,
           COUNT(DISTINCT report_date) AS distinct_days
    FROM {TABLES['fact_daily']}
""").df()
print(date_range)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating 1-row-per-client-content-day grain: 5
    earliest     latest  distinct_days
0 2025-01-27 2026-06-30            520


In [10]:
grain_check_full = con.sql(f"""
    SELECT COUNT(*) AS violating_groups
    FROM (
        SELECT client_hash_id, content_hash_id, report_date
        FROM {TABLES['fact_daily']}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").df()
print(grain_check_full)

total_rows = con.sql(f"SELECT COUNT(*) AS n FROM {TABLES['fact_daily']}").df()
print(total_rows)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   violating_groups
0              6390
          n
0  78835655


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [11]:
schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}").df()
print(schema.to_string(index=False))

             column_name column_type null  key default extra
             report_date        DATE  YES None    None  None
          client_hash_id     VARCHAR  YES None    None  None
         content_hash_id     VARCHAR  YES None    None  None
          client_has_gsc     BOOLEAN  YES None    None  None
          client_has_ga4     BOOLEAN  YES None    None  None
      gsc_data_available     BOOLEAN  YES None    None  None
      ga4_data_available     BOOLEAN  YES None    None  None
         gsc_impressions      BIGINT  YES None    None  None
              gsc_clicks      BIGINT  YES None    None  None
        gsc_sum_position      BIGINT  YES None    None  None
        gsc_avg_position      DOUBLE  YES None    None  None
           ga4_pageviews      BIGINT  YES None    None  None
            ga4_sessions      BIGINT  YES None    None  None
               ga4_users      BIGINT  YES None    None  None
    ga4_engaged_sessions      BIGINT  YES None    None  None
ga4_total_engagement_sec

In [12]:
# Core features (available for ~100% of rows, GSC-based)
feature_cols_core = ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "gsc_sum_position", "scroll_events"]

# GA4-dependent features (only available for ~3.6% of rows — usable only as optional enrichment)
feature_cols_ga4_optional = ["ga4_pageviews", "ga4_sessions", "ga4_users",
                               "ga4_engaged_sessions", "ga4_total_engagement_sec",
                               "sessions_organic", "sessions_direct", "sessions_referral",
                               "sessions_social", "sessions_paid", "sessions_ai"]

context_cols = ["report_date", "client_hash_id", "content_hash_id", "month"]

excluded_cols = ["ai_chatgpt", "ai_perplexity", "ai_gemini", "ai_copilot",
                   "ai_claude", "ai_meta", "ai_other",
                   "client_has_gsc", "client_has_ga4", "gsc_data_available", "ga4_data_available"]

print(f"Core feature columns (near-universal coverage): {len(feature_cols_core)}")
print(f"GA4-optional feature columns (only 3.6% coverage): {len(feature_cols_ga4_optional)}")
print(f"Context columns: {len(context_cols)}")
print(f"Excluded columns: {len(excluded_cols)}")
print(f"Label: is_declining (derived, not a raw column)")

Core feature columns (near-universal coverage): 5
GA4-optional feature columns (only 3.6% coverage): 11
Context columns: 4
Excluded columns: 11
Label: is_declining (derived, not a raw column)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# Claim 1: gsc_sum_position is redundant with gsc_avg_position (sum = avg * count of days with impressions... but actually check the raw relationship)
redundancy_check = con.sql(f"""
    SELECT corr(gsc_sum_position, gsc_avg_position) AS correlation
    FROM {TABLES['fact_daily']}
    WHERE gsc_impressions > 0
""").df()
print("Claim: gsc_sum_position is redundant with gsc_avg_position")
print(redundancy_check)

# Claim 2: AI referral columns are sparse
ai_sparsity = con.sql(f"""
    SELECT
        AVG(CASE WHEN ai_chatgpt > 0 THEN 1 ELSE 0 END) AS pct_rows_with_chatgpt,
        AVG(CASE WHEN sessions_ai > 0 THEN 1 ELSE 0 END) AS pct_rows_with_any_ai
    FROM {TABLES['fact_daily']}
""").df()
print("\nClaim: AI referral columns are sparse")
print(ai_sparsity)

# Claim 3: not every row has GA4 data (context flags matter)
ga4_coverage = con.sql(f"""
    SELECT
        AVG(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS pct_rows_with_ga4
    FROM {TABLES['fact_daily']}
""").df()
print("\nClaim: not all rows have GA4 data")
print(ga4_coverage)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Claim: gsc_sum_position is redundant with gsc_avg_position
   correlation
0     0.069844


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Claim: AI referral columns are sparse
   pct_rows_with_chatgpt  pct_rows_with_any_ai
0               0.000247              0.000383


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Claim: not all rows have GA4 data
   pct_rows_with_ga4
0           0.035726


Every claim from Section 2 is checked here with a real query rather than assumed. Two of the three original claims did not hold up: gsc_sum_position turned out to have only a 0.07 correlation with gsc_avg_position (not redundant, moved back into features), and GA4 data was found to be available for only 3.6% of rows, far more limited than assumed, requiring ga4_* and channel-split sessions_* columns to be reclassified as optional/enrichment rather than core features. The AI-referral sparsity claim held up and was even more extreme than expected (well under 0.1% of rows). This verification step directly changed the field contract in Section 2 and fed into the data limits documented in Section 4.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell you:

Why a page's ranking or traffic changed, only that it did. No causal signal (algorithm updates, competitor actions, seasonal demand) is captured.
Full-funnel behavior for 96.4% of rows, since GA4 data is only available for 3.6% of rows (verified in Section 3). Any model relying on ga4_* or channel-split sessions_* features effectively only generalizes to a small, non-representative subset of clients.
A complete history for most clients. As shown in Notebook 03's dim_clients check, only 4 of 104 clients have 12+ months of GSC history, most clients' earliest data starts well after the dataset's overall 2025-01-27 start date. A model trained assuming uniform history depth would silently underperform or bias toward clients with longer histories.
Perfectly unique daily rows. A small number of client+content+day groups (6,390 of 78.8M, ~0.008%, verified in Section 1) violate strict one-row-per-day grain, meaning naive per-day joins could occasionally double-count without deduplication.
Real AI-assistant referral behavior at scale, since ai_chatgpt/ai_perplexity/etc. columns have referral traffic in well under 0.1% of rows (verified in Section 3), any conclusion about AI-search impact would be built on a tiny, likely unrepresentative sample.
Client identity or content topic, since everything is hashed/anonymized (client_hash_id, content_hash_id), the data can show that something happened but not what the content was about or who the client is, by design.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.